# Example 15.6 / 4: Review a Generated Model Before Trusting It

Companion notebook for **15.6 Talks on AI for Optimization** in the AMPL lab Documentation, section "Review a Generated Model Before Trusting It". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Review a Generated Model Before Trusting It

In December 2022 the AMPL team asked ChatGPT to write AMPL models. Their notebook reports that the
answers were not always right, that ChatGPT corrected lines when shown the errors, and that it
sometimes worked flawlessly. The gift model it wrote assigns one gift to each person to maximize
total happiness within a budget:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PEOPLE;                                    # people receiving gifts
set GIFTS;                                     # gift types
param cost {GIFTS};                            # dollars per gift
param happiness {PEOPLE, GIFTS};               # happiness score of a gift for a person
param budget;                                  # dollars available

var x {PEOPLE, GIFTS} binary;                  # 1 if person p receives gift g

maximize total_happiness: sum {p in PEOPLE, g in GIFTS} happiness[p,g] * x[p,g];

# total cost of the gifts within the budget
subject to budget_constraint: sum {p in PEOPLE, g in GIFTS} cost[g] * x[p,g] <= budget;
# each person receives exactly one gift
subject to one_gift_per_person {p in PEOPLE}: sum {g in GIFTS} x[p,g] = 1;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
import pandas as pd

ampl.set["PEOPLE"] = ["Alice", "Bob", "Carol", "Dave", "Eve"]
ampl.set["GIFTS"] = ["Book", "Toy", "Chocolate", "Wine", "Flowers"]
ampl.param["cost"] = {"Book": 10, "Toy": 20, "Chocolate": 5, "Wine": 15, "Flowers": 7}
ampl.param["budget"] = 50
ampl.param["happiness"] = pd.DataFrame(
    {"Book": [3, 2, 5, 1, 4], "Toy": [5, 2, 4, 3, 1], "Chocolate": [1, 3, 4, 5, 2],
     "Wine": [2, 5, 3, 4, 1], "Flowers": [4, 3, 1, 2, 5]},
    index=["Alice", "Bob", "Carol", "Dave", "Eve"],
)

In [5]:
# run.py
ampl.solve(solver="highs")
print(ampl.solve_result, ampl.obj["total_happiness"].value())
print(ampl.get_data("{p in PEOPLE, g in GIFTS: x[p,g] > 0.5} cost[g]").to_dict())

HiGHS 1.14.0: optimal solution; objective 24
1 simplex iterations
1 branching nodes
solved 24.0
{('Alice', 'Flowers'): 7, ('Bob', 'Wine'): 15, ('Carol', 'Book'): 10, ('Dave', 'Chocolate'): 5, ('Eve', 'Flowers'): 7}


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set PEOPLE := Alice Bob Carol Dave Eve;
set GIFTS := Book Toy Chocolate Wine Flowers;
param cost := Book 10  Toy 20  Chocolate 5  Wine 15  Flowers 7;
param budget := 50;
param happiness:  Book  Toy  Chocolate  Wine  Flowers :=
Alice                3    5          1     2        4
Bob                  2    2          3     5        3
Carol                5    4          4     3        1
Dave                 1    3          5     4        2
Eve                  4    1          2     1        5;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display total_happiness, {p in PEOPLE, g in GIFTS: x[p,g] > 0.5} cost[g];

HiGHS 1.14.0: optimal solution; objective 24
1 simplex iterations
1 branching nodes
total_happiness = 24

cost[g] :=
Alice Flowers      7
Bob   Wine        15
Carol Book        10
Dave  Chocolate    5
Eve   Flowers      7
;

